In [0]:
# Create the landing folder inside your new volume
dbutils.fs.mkdirs("/Volumes/workspace/dev_bronze_layer/raw_files/landing")

In [0]:
# Filename: src/notebooks/bronze_ingestion.py
from pyspark.sql.functions import current_timestamp, input_file_name

# 1. Define the landing path in Volume
landing_path = "/Volumes/workspace/dev_bronze_layer/raw_files/landing/"
checkpoint_path = "/Volumes/workspace/dev_bronze_layer/raw_files/_checkpoints/bronze/"

# 2. Use Auto Loader (cloudFiles) to ingest data incrementally
(spark.readStream
  .format("cloudFiles")
  .option("cloudFiles.format", "csv")
  .option("cloudFiles.schemaLocation", checkpoint_path) # Infers and tracks schema
  .option("header", "true")
  .load(landing_path)
  .withColumn("ingestion_timestamp", current_timestamp()) # Audit column
  .withColumn("source_file", input_file_name())           # Audit column
  .writeStream
  .option("checkpointLocation", checkpoint_path)
  .trigger(availableNow=True) # Processes all new data and stops
  .toTable("workspace.dev_bronze_layer.events_raw"))